In [1]:
cd /home/sebastin/working/lab_lucac/sebastiN/projects/OCD_Symphony/

/home/sebastin/working/lab_lucac/sebastiN/projects/OCD_Symphony


In [23]:
import importlib
import OCD_symphony
import os
import numpy as np
import pandas as pd
import scipy
import scipy.stats
import torch
import seaborn as sbn
import sklearn
from matplotlib import pyplot as plt
from time import time

from OCD_symphony.utils import proj_dir, today
import OCD_symphony.data_preparation

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [24]:
importlib.reload(OCD_symphony)
importlib.reload(OCD_symphony.data_preparation)
importlib.reload(OCD_symphony.data_preparation.make_hcp_dataset)

# Load HCP dataset and demographics
t0 = time()
dataset, subjs = OCD_symphony.data_preparation.make_hcp_dataset.make_hcp_dataset_from_pt(path=os.path.join(proj_dir, 'data/hcp'))
hcp_age, hcp_gender = OCD_symphony.data_preparation.make_hcp_dataset.load_hcp_age_gender(subjs)
print(f"Lazy Loading of dataset done in {time()-t0:.1f}s, i.e. {(time()-t0)/60:.1f} minutes.")

/home/sebastin/miniconda3/envs/ocd-symphony/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/home/sebastin/miniconda3/envs/ocd-symphony/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/home/sebastin/miniconda3/envs/ocd-symphony/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed

Number of samples: (24717,), unique subjects: 1066, avg samples per subject: 23.186679174484052
Lazy Loading of dataset done in -356.7s.


In [25]:
n_samples = len(subjs)
output_file = os.path.join("/home/sebastin/tmp", "memmap_brain_symphony_inference_HCP_full.npz")

In [26]:
import OCD_symphony.model.run_inference
importlib.reload(OCD_symphony.model.model_components)
importlib.reload(OCD_symphony.model.run_inference)

t1 = time()
results = OCD_symphony.model.run_inference.run_inference(dataset, output_file, n_samples)
print(f"Inference took {time()-t1:.1f}s, i.e. {(time()-t1)/60:.1f} minutes.")

ValueError: Unknown format code 's' for object of type 'float'

In [28]:
importlib.reload(OCD_symphony.predictive_analysis.crossval_preproc)
t2 = time()
OCD_symphony.predictive_analysis.crossval_preproc.perform_classification_with_mean(inputs=results['all_roi_embeddings'],
                                                       subjs=subjs,
                                                       labels=np.array([0 if 'M' in gender else 1 for gender in hcp_gender]) )
print(f"Classification took {time()-t2:.1f}s, i.e. {(time()-t2)/60:.1f} minutes.")

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best params: {'clf__C': 0.1, 'clf__l1_ratio': 0.1}
Training accuracy 	 = 	 0.521 ± 0.041
Test accuracy 		 = 	 0.527 ± 0.042
Classification took 44.7s, i.e. 0.7 minutes.


In [29]:
importlib.reload(OCD_symphony.predictive_analysis.crossval_preproc)
OCD_symphony.predictive_analysis.crossval_preproc.perform_classification_with_PCA(inputs=results['all_perceiver_attn'],
                                                       subjs=subjs,
                                                       labels=np.array([0 if 'M' in gender else 1 for gender in hcp_gender]) )

--- SPLIT 1 ---
Compute T-stat from training set...


/home/sebastin/miniconda3/envs/ocd-symphony/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:551: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)


Perform dimensionality reduction on training set...


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values